# 5. 流式调用
invoke 和 stream 有什么区别？

1. invoke() ：同步调用，在模型输出完成后一次性获取响应，对于输出文本很长的场景，用户体验不好。
2. stream() ：流式调用，实时返回响应片段。调用后，返回一个 迭代器(iterator) ，可以通过循环来实时处理每一个新生成的chunk内容块。
> 注意：流式输出依赖于模型供应商对于流式输出的支持。 部分模型不支持流式输出，会报错。

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_API_BASE = os.getenv("OPENAI_API_BASE")

model = init_chat_model(
    model="openai:gpt-6-luna",
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE
)

for chunk in model.stream("写一首七言律诗，总结大模型的发展"):
    print(chunk.text, end="", flush=True) # 逐token输出

《大模型兴》

算力奔流启智门，千层参数聚乾坤。  
语通百艺融千卷，意接群机越万村。  
多模并行开异境，长思渐进破疑云。  
仍须明辨真和伪，方使新知济世人。

In [3]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(verbose=True,override=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

for chunk in model.stream("写一首七言律诗，总结大模型的发展"):
    print(chunk.text, end="", flush=True) # 逐token输出

《七律·大模型发展》

神经初醒问穹苍，  
数据洪流汇海洋。  
架构创新开境界，  
模型预训破苍茫。  
规模大小开新智，  
万象图文共显扬。  
治理对齐须并重，  
人机共进启新航。

stream()方式的优点：
1. 响应速度更快 — 用户不必等待完整输出
2. 交互体验更流畅 — 尤其在长文本或复杂推理场景下
3. 可实时展示模型思考过程

# 6. 批量调用

`batch()` 接收多个独立输入，默认在客户端通过线程池并发执行多次 `invoke()`，最后返回结果列表。它不会把这些输入合成一次对话，也不是模型厂商的离线 Batch API。

与逐个顺序调用相比，并发通常能缩短整批任务的总耗时，但请求数量并没有减少，也不会自动获得 token 费用折扣。实际耗时还取决于模型生成速度、网络和服务端限流。

适用场景：文档摘要、批量问答、数据预处理、多样本分类等。下面使用 `config={"max_concurrency": 2}` 限制最多同时执行两个请求。

参考：[LangChain 批量调用文档](https://docs.langchain.com/oss/python/langchain/models#batch)。

## 6.1 一次性接收所有响应

`batch()` 等待所有请求处理完毕，按原始输入顺序返回结果列表。

下面各个代码单元格都先加载 `conf/.env`，再读取配置、创建模型，可以单独运行。`from dotenv import load_dotenv` 只是导入函数，必须调用 `load_dotenv(...)` 才会把配置读入环境变量。

使用 `os.environ["变量名"]` 读取必填配置：变量缺失时会直接提示变量名，避免把 `None` 传给模型。示例沿用本项目的 `.env` 绝对路径，移动项目后需要修改这个路径。

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(
    dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env",
    override=True,
)
DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
DEEPSEEK_BASE_URL = os.environ["DEEPSEEK_BASE_URL"]

model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

messages = [
    "请用一句话介绍一下你自己",
    "中国的首都是哪里?",
    "日本的首都是哪里?",
]

results = model.batch(messages, config={"max_concurrency": 2})
print(type(results))  # list[AIMessage]
for question, result in zip(messages, results):
    print(f"问题：{question}")
    print(f"回答：{result.text}\n")  # 只显示正文；print(result) 会显示全部元数据

<class 'list'>
问题：请用一句话介绍一下你自己
回答：我是由深度求索公司创造的 DeepSeek AI 助手，可以为你答疑解惑、处理信息并协助完成多种任务。

问题：中国的首都是哪里?
回答：中国的首都是**北京**。

问题：日本的首都是哪里?
回答：日本的首都是**东京**。  
通常指**东京都**，是日本中央政府、国会和天皇居所所在地。



In [13]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv(
    dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env",
    override=True,
)
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

messages = [
    "请用一句话介绍一下你自己",
    "中国的首都是哪里?",
    "日本的首相是谁?",
]

results = model.batch(messages, config={"max_concurrency": 2})
print(type(results))
for question, result in zip(messages, results):
    print(f"问题：{question}")
    print(f"回答：{result.text}\n")

<class 'list'>
问题：请用一句话介绍一下你自己
回答：我是 Codex，一个基于 GPT-5 的 AI 编程助手，可以帮你阅读、编写和调试代码。

问题：中国的首都是哪里?
回答：中国的首都是北京。

问题：日本的首相是谁?
回答：截至 2026 年 8 月，日本首相是高市早苗。



## 6.2 按完成顺序接收响应

`batch_as_completed()` 在每个请求完成后立即产出该请求的完整结果，不必等整批结束。结果可能乱序，每项都是 `(index, response)` 元组，`index` 是原始输入的索引，可以用来找到对应问题或恢复顺序。

这里的“逐个返回”指的是逐个返回完整回答，与 `stream()` 逐块返回同一个回答不同。

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(
    dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env",
    override=True,
)
DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
DEEPSEEK_BASE_URL = os.environ["DEEPSEEK_BASE_URL"]

model = init_chat_model(
    model="deepseek:deepseek-v4-pro",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

messages = [
    "请用一句话介绍一下你自己",
    "中国的首都是哪里?",
    "日本的首相是谁?",
]

results = model.batch_as_completed(messages, config={"max_concurrency": 2})
print(type(results))  # generator，遍历时逐个取得已完成的结果
for index, result in results:
    print(f"输入索引：{index}，问题：{messages[index]}")
    print(f"回答：{result.text}\n")

<class 'generator'>


输入索引：1，问题：中国的首都是哪里?
回答：中国的首都是北京。



输入索引：2，问题：日本的首都是哪里?
回答：日本的首都是**东京**。



输入索引：0，问题：请用一句话介绍一下你自己
回答：我是DeepSeek，由深度求索公司开发的AI助手，致力于用热情和细腻的方式为你提供帮助。



## 6.3 性能对比

下面两个单元格使用相同模型、相同参数和相同输入，分别测量批量并发与顺序调用的总耗时。示例关闭 DeepSeek 思考模式，并要求只输出译文，以减少额外生成内容对演示的干扰。

`time.perf_counter()` 适合测量一段程序的耗时。批量示例最多并发 2 个请求；顺序示例每次等上一条完成后再发下一条。

一次测量只用于理解调用方式，不能作为严格的性能结论：网络波动、输出长度、缓存命中和服务端负载都会影响结果，批量调用也不保证每次都更快。

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(
    dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env",
    override=True,
)
DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
DEEPSEEK_BASE_URL = os.environ["DEEPSEEK_BASE_URL"]

import time

model = init_chat_model(
    model="deepseek:deepseek-v4-pro",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}},
)
inputs = [
    "翻译成英文，只输出译文：春天来了",
    "翻译成英文，只输出译文：夏天很热",
    "翻译成英文，只输出译文：秋天落叶",
    "翻译成英文，只输出译文：冬天下雪",
]

start = time.perf_counter()
responses = model.batch(inputs, config={"max_concurrency": 2})
batch_time = time.perf_counter() - start

print("批量调用结果：")
for i, response in enumerate(responses, start=1):
    print(f"{i}. {response.text}")
print(f"批量调用总耗时：{batch_time:.2f} 秒")

批量调用结果：
1. Spring has come.
2. Summer is very hot.
3. Autumn leaves fall.
4. It snows in winter.
批量调用总耗时：3.46 秒


循环调用 `invoke()`：按顺序处理相同的四个输入。总耗时在所有调用结束后计算和打印。

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(
    dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env",
    override=True,
)
DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
DEEPSEEK_BASE_URL = os.environ["DEEPSEEK_BASE_URL"]

import time

model = init_chat_model(
    model="deepseek:deepseek-v4-pro",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}},
)
inputs = [
    "翻译成英文，只输出译文：春天来了",
    "翻译成英文，只输出译文：夏天很热",
    "翻译成英文，只输出译文：秋天落叶",
    "翻译成英文，只输出译文：冬天下雪",
]

start = time.perf_counter()
loop_responses = []
for prompt in inputs:
    response = model.invoke(prompt)
    loop_responses.append(response)
loop_time = time.perf_counter() - start

print("循环调用结果：")
for i, response in enumerate(loop_responses, start=1):
    print(f"{i}. {response.text}")
print(f"循环调用总耗时：{loop_time:.2f} 秒")

循环调用结果：
1. Spring has arrived.
2. Summer is very hot.
3. Autumn leaves fall.
4. It snows in winter.
循环调用总耗时：7.96 秒


## 6.4 本次报错的原因与排查方法

| 现象 | 原因 | 处理方式 |
|---|---|---|
| `ChatDeepSeek` 的 `base_url` 报 `input_value=None` | 创建模型时传入的地址是 `None`；原批量单元格没有加载 `.env`，单独运行或重启内核后不能依赖前面残留的环境变量 | 先调用 `load_dotenv(...)`，再读取 `DEEPSEEK_API_KEY` 和 `DEEPSEEK_BASE_URL` |
| `403`，提示“该令牌无权访问模型 gpt-5.5” | 服务端拒绝当前令牌访问该模型，不是 `batch()` 语法错误 | 使用当前服务和令牌支持的模型，或向服务提供方申请权限；本例改为已验证可用的 `gpt-6-luna` |
| 读取 `OPENAI_BASE_URL` 得到 `None` | 本项目 `.env` 实际配置的是 `OPENAI_API_BASE`，两个名字不同 | 读取 `OPENAI_API_BASE`，并显式传给 `base_url` |

> 这几个问题应分别处理。403 已明确说明模型权限不足；环境变量名不一致是另一个配置问题，不能把所有错误都归因于并发。

Jupyter 会保存旧输出，修改代码不会自动删除旧报错。修正后需要重新运行对应单元格；如果内核状态混乱，可以重启内核后重新运行。

批量调用默认遇到异常会抛出。如果希望单条请求失败时仍能检查其他请求的结果，可以使用：

```python
results = model.batch(
    messages,
    config={"max_concurrency": 2},
    return_exceptions=True,
)
for index, result in enumerate(results):
    if isinstance(result, Exception):
        print(f"第 {index} 条失败：{type(result).__name__}: {result}")
    else:
        print(f"第 {index} 条成功：{result.text}")
```

`batch_as_completed()` 也支持 `return_exceptions=True`，对解包出的 `result` 做相同检查即可。这个参数只是让错误作为结果返回，不会修复缺失配置或模型权限；模型初始化阶段的错误仍需先解决。